In [50]:
import json
import re
import time
import csv
import statistics
import subprocess
import shutil
import platform
import os
from pathlib import Path

import numpy as np
import requests

PROJECT_DIR = Path.cwd()
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def resolve_ollama_binary() -> str | None:
    in_path = shutil.which("ollama")
    if in_path:
        return in_path

    common_paths = [
        "/usr/local/bin/ollama",
        "/usr/bin/ollama",
    ]
    for p in common_paths:
        if Path(p).exists():
            return p
    return None


def ensure_ollama_installed() -> str:
    bin_path = resolve_ollama_binary()
    if bin_path:
        return bin_path

    system = platform.system().lower()
    if system != "linux":
        raise RuntimeError(
            "Ollama is not installed and auto-install is only enabled for Linux in this notebook. "
            "Install Ollama from https://ollama.com/download and rerun Cell 1."
        )

    print("Ollama not found. Installing prerequisites...")
    subprocess.run(
        ["bash", "-lc", "apt-get update -y && apt-get install -y curl zstd ca-certificates"],
        check=True,
        text=True,
    )

    print("Installing Ollama...")
    install_cmd = "curl -fsSL https://ollama.com/install.sh | OLLAMA_NO_SYSTEMD=1 sh"
    install_proc = subprocess.run(
        ["bash", "-lc", install_cmd],
        check=False,
        text=True,
        capture_output=True,
    )

    if install_proc.returncode != 0:
        print("Installer stdout:\n", install_proc.stdout[-3000:])
        print("Installer stderr:\n", install_proc.stderr[-3000:])
        raise RuntimeError("Ollama install failed. See installer logs above.")

    # Ensure /usr/local/bin is in PATH for this kernel session.
    os.environ["PATH"] = os.environ.get("PATH", "") + ":/usr/local/bin"

    bin_path = resolve_ollama_binary()
    if not bin_path:
        raise RuntimeError("Ollama install completed but binary was not found")

    return bin_path


OLLAMA_BIN = ensure_ollama_installed()
subprocess.run([OLLAMA_BIN, "--version"], check=True, text=True)


def ollama_running() -> bool:
    try:
        _ = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False


if not ollama_running():
    log_path = PROJECT_DIR / "ollama.log"
    with open(log_path, "a", encoding="utf-8") as f:
        _proc = subprocess.Popen([OLLAMA_BIN, "serve"], stdout=f, stderr=f)
    time.sleep(3)
    print("Started Ollama server")
else:
    print("Ollama already running")

Started Ollama server


In [51]:
HARDCODED_RULES = [
    "Do not promise loan approval or guarantee outcomes.",
    "Do not request or expose sensitive personal data (full SSN, full bank account number, OTP, passwords).",
    "Always include a neutral compliance disclaimer: This is general information, not financial advice.",
]

BORROWER_SYSTEM = """You are a borrower support assistant.
Write a concise, helpful answer to the borrower query.
Keep tone empathetic and practical.
"""

VALIDATOR_SYSTEM = """You are a strict compliance validator.
You will receive hardcoded rules and an assistant response.
Return ONLY JSON with this exact shape:
{
  \"is_compliant\": true/false,
  \"violated_rules\": [\"...\"],
  \"fixed_response\": \"...\"
}
"""
JSON_REPAIR_SYSTEM = """Convert the following text into valid JSON only.
Output one JSON object with exactly these keys: is_compliant, violated_rules, fixed_response.
Do not include markdown code fences or extra text.
"""



In [52]:
!pip install ollama


In [53]:
MODEL_NAME = MODEL
print(f"MODEL_NAME set to: {MODEL_NAME}")

MODEL_NAME set to: qwen2.5:7b


In [54]:
print(f"Pulling model {MODEL_NAME}...")
subprocess.run([OLLAMA_BIN, "pull", MODEL_NAME], check=True, text=True)
print("Model pulled successfully.")

Pulling model qwen2.5:7b...
Model pulled successfully.


In [55]:
def ollama_generate(prompt: str, temperature: float = 0.0) -> str:
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature},
    }
    response = requests.post("http://127.0.0.1:11434/api/generate", json=payload, timeout=120)
    response.raise_for_status()
    return response.json().get("response", "").strip()

def safe_json_extract(text: str) -> dict:
    """Extracts the first JSON object from a string, or returns an empty dict if none found."""
    try:
        # Regular expression to find JSON objects
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        if match:
            json_str = match.group(0)
            return json.loads(json_str)
    except json.JSONDecodeError:
        pass  # Malformed JSON, continue to try alternative methods or return empty
    return {} # Fallback if no valid JSON or regex fails


def build_borrower_prompt(query: str) -> str:
    return (
        BORROWER_SYSTEM
        + "\nBorrower query:\n"
        + query
        + "\n\nInclude the exact sentence if missing: This is general information, not financial advice."
    )


def build_validator_prompt(response_text: str) -> str:
    rules = "\n".join([f"- {r}" for r in HARDCODED_RULES])
    return (
        VALIDATOR_SYSTEM
        + "\nRules:\n"
        + rules
        + "\n\nAssistant response to validate:\n"
        + response_text
    )


def run_two_agent_turn(query: str) -> dict:
    start = time.perf_counter()

    borrower_response = ollama_generate(build_borrower_prompt(query), temperature=0.2)

    validator_raw = ollama_generate(build_validator_prompt(borrower_response), temperature=0.0)
    try:
        validator_json = safe_json_extract(validator_raw)
    except Exception:
        repaired = ollama_generate(JSON_REPAIR_SYSTEM + "\nText to convert:\n" + validator_raw, temperature=0.0)
        validator_json = safe_json_extract(repaired)

    validator_json.setdefault("is_compliant", False)
    validator_json.setdefault("violated_rules", [])
    validator_json.setdefault("fixed_response", borrower_response)

    latency = time.perf_counter() - start
    return {
        "query": query,
        "borrower_response": borrower_response,
        "validator": validator_json,
        "latency_seconds": latency,
    }

In [56]:
!pip install pandas

In [ ]:
import pandas as pd
import json
import statistics

N_RUNS = 10 # Define N_RUNS with a default value

borrower_queries = [
    "Can I get a $12,000 personal loan with low monthly payments?",
    "I have a credit score around 610. What loan options might I have?",
    "Can you prequalify me for a $25,000 debt consolidation loan?",
    "I am self-employed. Can I still apply for an unsecured loan?",
    "What documents do I need for a personal loan application?",
]

all_runs = []
per_query_summary = []
agent_outputs = []

for borrower_query in borrower_queries:
    runs = []
    for i in range(N_RUNS):
        run_result = run_two_agent_turn(borrower_query)
        run_result["run_index"] = i + 1
        runs.append(run_result)
        all_runs.append(run_result)

    e2e = [r["latency_seconds"] for r in runs] # Convert to ms
    borrower_lat = [r["latency_seconds"] for r in runs]
    validator_lat = [r["latency_seconds"] for r in runs]

    sorted_e2e = sorted(e2e)
    p95_idx = max(0, min(len(sorted_e2e) - 1, int(0.95 * len(sorted_e2e)) - 1))
    p95_e2e = sorted_e2e[p95_idx]

    per_query_summary.append(
        {
            "borrower_query": borrower_query,
            "n_runs": N_RUNS,
            "avg_end_to_end_ms": round(statistics.mean(e2e), 2),
            "median_end_to_end_ms": round(statistics.median(e2e), 2),
            "p95_end_to_end_ms": round(p95_e2e, 2),
            "avg_borrower_agent_ms": round(statistics.mean(borrower_lat), 2),
            "avg_validator_agent_ms": round(statistics.mean(validator_lat), 2),
        }
    )

    latest = runs[-1]
    validator_report = latest.get("validator", {})
    agent_outputs.append(
        {
            "borrower_query": borrower_query,
            "generator_given": latest.get("borrower_response", ""),
            "validator_saying": {
                "is_compliant": validator_report.get("is_compliant"),
                "violated_rules": validator_report.get("violated_rules"),
                "fixed_response": validator_report.get("fixed_response", ""),
            },
        }
    )

all_e2e = [r["latency_seconds"] for r in all_runs]
all_sorted_e2e = sorted(all_e2e)
all_p95_idx = max(0, min(len(all_sorted_e2e) - 1, int(0.95 * len(all_sorted_e2e)) - 1))
all_p95_e2e = all_sorted_e2e[all_p95_idx]

summary = {
    "benchmark": {
        "n_queries": len(borrower_queries),
        "runs_per_query": N_RUNS,
        "total_runs": len(all_runs),
        "overall_avg_end_to_end_ms": round(statistics.mean(all_e2e), 2),
        "overall_median_end_to_end_ms": round(statistics.median(all_e2e), 2),
        "overall_p95_end_to_end_ms": round(all_p95_e2e, 2),
    },
    "per_query_summary": per_query_summary,
    "agent_outputs": agent_outputs,
}

# Convert to DataFrames before saving
pd.DataFrame(per_query_summary).to_csv(RESULTS_DIR / "per_query_summary.csv", index=False)
pd.DataFrame(agent_outputs).to_csv(RESULTS_DIR / "agent_outputs.csv", index=False)
# For the overall summary, we can save it as JSON or a single-row CSV
with open(RESULTS_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, ensure_ascii=True, indent=2))